# **Perguntas**



*   O el niño afeta o índice de chuvas no no RN?
*  Existe uma correlação entre a intensidade do el nino e a intensidade das chuvas?

*   Há deslocamento do período chuvoso com a intensificação do El Niño?
*   Existe uma correlação entre a temperatura e a intensidade do El Ninõ?





In [ ]:
import subprocess, sys

DEPS = [
    ('pandas',            'pandas'),
    ('numpy',             'numpy'),
    ('requests',          'requests'),
    ('matplotlib',        'matplotlib'),
    ('seaborn',           'seaborn'),
    ('pyarrow',           'pyarrow'),
    ('epiweeks',          'epiweeks'),
    ('scipy',             'scipy'),
    ('statsmodels',       'statsmodels'),
    ('openmeteo-requests','openmeteo_requests'),
    ('requests-cache',    'requests_cache'),
    ('retry-requests',    'retry_requests'),
]

print(' Verificando dependências...')
for pkg, imp_name in DEPS:
    try:
        __import__(imp_name)
        print(f'  ✔ {pkg}')
    except ImportError:
        print(f'  Instalando {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f'  ✔ {pkg} instalado')


 Verificando dependências...
  ✔ pandas
  ✔ numpy
  ✔ requests
  ✔ matplotlib
  ✔ seaborn
  ✔ pyarrow
  Instalando epiweeks...
  ✔ epiweeks instalado
  ✔ scipy
  ✔ statsmodels
  Instalando openmeteo-requests...
  ✔ openmeteo-requests instalado
  Instalando requests-cache...
  ✔ requests-cache instalado
  Instalando retry-requests...
  ✔ retry-requests instalado


In [ ]:
# 1. Bibliotecas padrão do Python
import gc
import io
import os
import sys
import time
import warnings
from datetime import datetime, date
from pathlib import Path

# 2. Manipulação e análise de dados
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# 3. Coleta de dados
import requests
import zipfile
import openmeteo_requests
import requests_cache
from retry_requests import retry
import unicodedata

# 4. Visualização de dados
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch as _Patch

# 5. Estatística, datas epidemiológicas e outros utilitários
import datetime as _dt
from epiweeks import Week
from scipy import stats
from scipy.stats import spearmanr

# 6. Statsmodels (usado nas análises complementares)
try:
    from statsmodels.tsa.seasonal import seasonal_decompose
except ImportError:
    import subprocess, sys as _sys
    subprocess.check_call([_sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'])
    from statsmodels.tsa.seasonal import seasonal_decompose

# 7. Google Colab (para montar o drive)
from google.colab import drive


# **Definir o período que será analisado**

In [ ]:
#ANOS        = list(range(2003, 2023))
#DATA_INICIO = '2003-01-01'
#DATA_FIM    = '2022-12-31'

# **Definir as fontes escolhidas**

In [ ]:
# -- URLs --
# El Niño
URL_EL_NINO = 'https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt'

# Dados climáticos
# É o mesmo que usamos no artigo anterior, mas pode ser que tenha outro melhor
URL_OPENMETEO_HIST = 'https://archive-api.open-meteo.com/v1/archive'

In [ ]:
# Carregamos os dados do El Niño diretamente da NOAA se ainda não estiverem definidos
try:
    df_el_nino
except NameError:
    df_el_nino = pd.read_csv(URL_EL_NINO, sep='\\s+', header=0)

df_el_nino.to_csv("dados_El_nino")
# Criamos uma cópia para preservar os dados brutos antes das alterações
df_el_nino_pre_processamento = df_el_nino.copy()

# Definimos as condições lógicas para as anomalias
cond_quente = df_el_nino['ANOM'] >= 0.5
cond_frio = df_el_nino['ANOM'] <= -0.5

# Para rastrear sequências consecutivas de pelo menos 5 trimestres
def classificar_eventos(df, limite_consecutivos=5):
    n = len(df)
    classificacao = ['Neutro'] * n

    # Inicializamos contadores para sequências quentes e frias
    seq_quente = 0
    seq_fria = 0

    # Primeiro passo: identificar onde ocorrem sequências de pelo menos 5 trimestres
    for i in range(n):
        anom = df.loc[i, 'ANOM']
        if anom >= 0.5:
            seq_quente += 1
            seq_fria = 0
        elif anom <= -0.5:
            seq_fria += 1
            seq_quente = 0
        else:
            seq_quente = 0
            seq_fria = 0

        # Se atingiu o limite de persistência, marca o período atual e os anteriores correspondentes
        if seq_quente >= limite_consecutivos:
            for j in range(i - seq_quente + 1, i + 1):
                classificacao[j] = 'El Niño'
        elif seq_fria >= limite_consecutivos:
            for j in range(i - seq_fria + 1, i + 1):
                classificacao[j] = 'La Niña'

    return classificacao

# Aplica a função de classificação oficial diretamente no df_el_nino
df_el_nino['FASE'] = classificar_eventos(df_el_nino)

# Exibe o resultado das contagens de cada fase histórica
print("\nDistribuição das Fases Climáticas no Histórico Oficial da NOAA (df_el_nino):")
print(df_el_nino['FASE'].value_counts())

# Amostra dos dados classificados
display(df_el_nino.tail(20))


Distribuição das Fases Climáticas no Histórico Oficial da NOAA (df_el_nino):
FASE
Neutro     484
La Niña    221
El Niño    215
Name: count, dtype: int64


,SEAS,YR,TOTAL,ANOM,FASE
900,DJF,2025,26.09,-0.46,Neutro
901,JFM,2025,26.56,-0.22,Neutro
902,FMA,2025,27.15,-0.08,Neutro
903,MAM,2025,27.65,0.02,Neutro
904,AMJ,2025,27.76,-0.04,Neutro
905,MJJ,2025,27.61,-0.02,Neutro
906,JJA,2025,27.18,-0.11,Neutro
907,JAS,2025,26.70,-0.26,Neutro
908,ASO,2025,26.33,-0.43,Neutro
909,SON,2025,26.14,-0.57,Neutro


### Significado dos Atributos do Índice ONI (El Niño) df_el_nino

* **`SEAS` (Estação / Trimestre Móvel):** Representa o período de três meses consecutivos analisados. A sigla utiliza a primeira letra de cada mês em inglês. Por exemplo:
  * `DJF`: Dezembro, Janeiro e Fevereiro (*December, January, February*)
  * `JFM`: Janeiro, Fevereiro e Março (*January, February, March*)
  * `MAM`: Março, Abril e Maio (*March, April, May*)

* **`YR` (Ano):** Indica o ano correspondente à observação.

* **`TOTAL` (Temperatura Média Absoluta):** É a média real da temperatura da superfície do mar (em °C) medida na região do Niño 3.4 (zona central do Oceano Pacífico Equatorial).

* **`ANOM` (Anomalia Térmica):** É o indicador mais importante. Representa a diferença de temperatura em relação à média histórica normal para aquela época do ano:
  * **Valores ≥ +0.5 °C** por cinco períodos consecutivos indicam a ocorrência de **El Niño** (aquecimento incomum).
  * **Valores ≤ -0.5 °C** por cinco períodos consecutivos indicam a ocorrência de **La Niña** (resfriamento incomum).

Um evento de El Niño é definido quando o Índice Oceânico Niño Relativo (RONI) permanece igual ou superior a 0,5 °C por pelo menos 5 trimestres de acordo com o INMET https://portal.inmet.gov.br/noticias/el-ni%C3%B1o-cada-vez-mais-pr%C3%B3ximo

Se quisermos responder se a intensidade do El Ninõ afeta com a mesma intensidade as chuvas reduziriamos os dados a no máximo 215 linhas (as classificadas como El Niño). Esta observação é um ponto importantíssimo a ser considerado para definir o período de observação

In [ ]:
ANO_CORTE = 2000

# Filtrando o DataFrame a partir do ano de corte desejado e apenas com a fase 'El Niño'
df_filtrado = df_el_nino[(df_el_nino['YR'] >= ANO_CORTE) & (df_el_nino['FASE'] == 'El Niño')]

total_meses_filtrado = len(df_filtrado)
anos_unicos = df_filtrado['YR'].nunique()

print(f"A partir do ano {ANO_CORTE}:")
print(f"- Total de meses (linhas) classificados como El Niño: {total_meses_filtrado} meses")
print(f"- Distribuídos ao longo de {anos_unicos} anos distintos.")

if total_meses_filtrado > 0:
    print("\nResumo dos trimestres encontrados por ano:")
    display(df_filtrado.groupby('YR').size().to_frame(name='Quantidade de Meses (Trimestres)'))

A partir do ano 2000:
- Total de meses (linhas) classificados como El Niño: 76 meses
- Distribuídos ao longo de 16 anos distintos.

Resumo dos trimestres encontrados por ano:


,Quantidade de Meses (Trimestres)
YR,
2002,6
2003,2
2004,5
2005,2
2006,4
2007,1
2009,6
2010,3
2014,3


# **Definir método para representar o RN**

Poderiamos usar a técnica de municípios estratégicos do último artigo, ou a mediana da temperatura do RN, ou a média, etc.

# **Definir a medida que analisará a correlação**

Pode ser Spearman como no último artigo